In [0]:
df_trip = spark.read.option("header", "true").option("inferSchema", "true").csv("abfss://bronze@vedatalakestoragedemo.dfs.core.windows.net/demo_location/taxi_data/bronze/trip_data_1.csv")
display(df_trip)
display(df_trip.describe())

In [0]:
df_fare = spark.read.option("header", "true").option("inferSchema", "true").csv("abfss://bronze@vedatalakestoragedemo.dfs.core.windows.net/demo_location/taxi_data/bronze/trip_fare_1.csv")
display(df_fare)
display(df_fare.describe())

In [0]:
df_fare.printSchema()

In [0]:
def clean_dataset(df):
    df_duplicate = df.dropDuplicates()
    df_columns = df_duplicate.toDF(*(c.strip() for c in df_duplicate.columns))
    df_cleaned = df_columns.filter(df_columns.vendor_id != 'CMT')
    return df_cleaned


def prepare_df_fare(df):
    df_clean = clean_dataset(df)
    df_prep = df_clean.dropna(subset=['total_amount'])
    return df_prep

def prepare_df_trip(df):
    df_clean = clean_dataset(df)
    df_prep = df_clean.dropna(subset=['trip_distance'])
    return df_prep

def silver_save_as_delta(df, folder_name, base_path):
    try:
        folder_path = f"{base_path}/{folder_name}"
        delta_path = f"{folder_path}/{folder_name}_cleaned"

        # Save as Delta
        df.write.format("delta").mode("overwrite").save(delta_path)

        return delta_path
        
    except Exception as e:
        print(f"Error saving data: {e}")


def save_or_update_deltatable(df, folder_name, catalog="azure_cloud", schema="default"):
    table_name = f"{catalog}.{schema}.{folder_name}_cleaned"
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    return table_name

def bronze_to_silver_taxi():

    df_trip = spark.read.option("header", "true").option("inferSchema", "true").csv("abfss://bronze@vedatalakestoragedemo.dfs.core.windows.net/demo_location/taxi_data/bronze/trip_data_1.csv")

    df_fare = spark.read.option("header", "true").option("inferSchema", "true").csv("abfss://bronze@vedatalakestoragedemo.dfs.core.windows.net/demo_location/taxi_data/bronze/trip_fare_1.csv")

    df_trip_cleaned = prepare_df_trip(df_trip)
    df_fare_cleaned = prepare_df_fare(df_fare)

    # save as delta
    
    trip_name = "taxi_trip_data"
    fare_name = "taxi_fare_data"

    silver_file_path = "abfss://bronze@vedatalakestoragedemo.dfs.core.windows.net/demo_location/taxi_data/silver"
    df_trip_location = silver_save_as_delta(df_trip_cleaned, "trip_name", silver_file_path)
    df_fare_location =  silver_save_as_delta(df_trip_cleaned, "fare_name", silver_file_path)

    save_or_update_deltatable(df_trip_cleaned, trip_name)
    save_or_update_deltatable(df_fare_cleaned, fare_name)

    print(f"Data saved to {silver_file_path} and Delta table {fare_name}_cleaned and {trip_name}_cleaned.")

  




In [0]:
bronze_to_silver_taxi()

In [0]:
%sql
from azure_cloud.default.taxi_trip_data_cleaned select *

In [0]:
df_trip_cleaned = prepare_df_trip(df_trip)
df_trip_cleaned.printSchema()
df_trip_cleaned.display()



In [0]:
df_fare_cleaned = prepare_df_fare(df_fare)
df_fare_cleaned.printSchema()
df_fare_cleaned.display()